# LeetCode 498 — Diagonal Traverse bằng Python

Link đề: https://leetcode.com/problems/diagonal-traverse/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc (đề gốc có hình minh họa đường đi zigzag trên ma trận; trong notebook ta vẽ lại bằng bảng ký tự). Mọi đáp án được đối chiếu với một oracle độc lập (sắp xếp các ô theo khóa).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: duyệt ma trận `m x n` theo **các đường chéo phụ** (anti-diagonal), **xen kẽ** hướng đi lên-phải và đi xuống-trái, bắt đầu từ ô `(0, 0)`.
2. Nắm công cụ Python: chỉ số hai chiều `mat[i][j]`, `len(mat)` / `len(mat[0])`, `min` / `max`, danh sách và `reverse`, `defaultdict(list)`, `sorted(..., key=...)`, biểu thức điều kiện một dòng.
3. Đi từ **sắp xếp các ô theo khóa** → **gom theo `i + j` rồi đảo xen kẽ** → **mô phỏng bước đi và "bật" khi chạm biên** → **duyệt từng đường chéo với điểm bắt đầu tính bằng công thức**.
4. Hiểu mấu chốt: mọi ô trên cùng một đường chéo phụ có **cùng tổng `i + j = k`**, có đúng `m + n - 1` đường chéo `k = 0 .. m+n-2`; đường chéo chẵn đi **lên**, đường chéo lẻ đi **xuống**.
5. Tránh các lỗi kinh điển: đếm sai số đường chéo, kiểm tra biên **sai thứ tự** ở góc, tính điểm bắt đầu bằng sai chiều (`m` ↔ `n`), đảo sai đường chéo nào, tạo danh sách lồng bằng `[[]] * k` (cùng tham chiếu).

## 2. Đề bài & ví dụ

Cho ma trận `mat` kích thước `m x n`. Trả về một mảng chứa **mọi phần tử của ma trận theo thứ tự đường chéo**: bắt đầu ở góc trên-trái, đi theo đường chéo **lên-phải**, chạm biên thì đổi sang đường chéo kế tiếp và đi **xuống-trái**, rồi lại lên-phải, v.v. cho tới khi hết ma trận.

**Ví dụ 1:** `mat = [[1,2,3],[4,5,6],[7,8,9]]` → `[1,2,4,7,5,3,6,8,9]`.

**Ví dụ 2:** `mat = [[1,2],[3,4]]` → `[1,2,3,4]`.

**Ràng buộc:**

- `m == mat.length`, `n == mat[i].length`
- `1 <= m, n <= 10^4`
- `1 <= m * n <= 10^4`
- `-10^5 <= mat[i][j] <= 10^5`

In [1]:
import random
import time
from collections import defaultdict
from typing import List

examples = [
    ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [1, 2, 4, 7, 5, 3, 6, 8, 9]),
    ([[1, 2], [3, 4]], [1, 2, 3, 4]),
]
for mat, expected in examples:
    print(mat, "->", expected)

[[1, 2, 3], [4, 5, 6], [7, 8, 9]] -> [1, 2, 4, 7, 5, 3, 6, 8, 9]
[[1, 2], [3, 4]] -> [1, 2, 3, 4]


### 2.1. Đọc kỹ đề

- Ma trận ví dụ 1, với số thứ tự lần đến thăm (từ 1):

```
 1   2   6        thứ tự thăm của (i, j):
 3   5   7        1 = (0,0)   2 = (0,1)   3 = (1,0)   4 = (2,0)   5 = (1,1)
 4   8   9        6 = (0,2)   7 = (1,2)   8 = (2,1)   9 = (2,2)
```

  Giá trị theo thứ tự đó: `mat[0][0]=1, mat[0][1]=2, mat[1][0]=4, mat[2][0]=7, mat[1][1]=5, mat[0][2]=3, mat[1][2]=6, mat[2][1]=8, mat[2][2]=9`.
- Đường chéo thứ nhất `{(0,0)}` chỉ có 1 ô; đường chéo thứ hai `{(0,1),(1,0)}` đi **xuống-trái**: `(0,1)` rồi `(1,0)`; đường chéo thứ ba `{(0,2),(1,1),(2,0)}` đi **lên-phải**: `(2,0)`, `(1,1)`, `(0,2)`; ...
- `m * n <= 10^4` nên mọi thuật toán `O(m·n)` (và cả `O(m·n·log(m·n))`) đều rất nhanh. Nhưng `m` có thể là `1` và `n` là `10^4` (hoặc ngược lại): ma trận một hàng / một cột là **edge case** quan trọng.
- Ma trận **không nhất thiết vuông**.

## 3. Kiến thức nền

### 3.1. Ma trận là "danh sách các danh sách"

`mat[i][j]` là phần tử hàng `i`, cột `j` (đếm từ 0). `len(mat)` là số hàng `m`, `len(mat[0])` là số cột `n` (mọi hàng đều cùng độ dài trong bài này).

In [2]:
mat = [[1, 2, 3, 4],
       [5, 6, 7, 8],
       [9, 10, 11, 12]]
m, n = len(mat), len(mat[0])
print("m =", m, "n =", n)
print(mat[0][3], mat[2][0], mat[1][2])
for i in range(m):
    print([mat[i][j] for j in range(n)])

m = 3 n = 4
4 9 7
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]


### 3.2. Đường chéo phụ: cùng `i + j`

Khi đi **lên-phải** thì `i` giảm 1, `j` tăng 1 ⇒ **tổng `i + j` không đổi**. Khi đi **xuống-trái**: `i` tăng 1, `j` giảm 1 ⇒ tổng cũng không đổi. Vậy mỗi đường chéo phụ là tập các ô có cùng `i + j`.

In [3]:
for k in range(m + n - 1):
    cells = [(i, k - i) for i in range(m) if 0 <= k - i < n]
    print(f"k = {k}: {cells}")

k = 0: [(0, 0)]
k = 1: [(0, 1), (1, 0)]
k = 2: [(0, 2), (1, 1), (2, 0)]
k = 3: [(0, 3), (1, 2), (2, 1)]
k = 4: [(1, 3), (2, 2)]
k = 5: [(2, 3)]


### 3.3. Có bao nhiêu đường chéo? Vì sao `m + n - 1`?

`i + j` nhỏ nhất là `0 + 0 = 0`, lớn nhất là `(m-1) + (n-1) = m + n - 2`. Các giá trị `0, 1, ..., m+n-2` là **`m + n - 1`** số. Mỗi giá trị đều có ít nhất một ô nên có đúng `m + n - 1` đường chéo.

### 3.4. `min` / `max` để tính điểm bắt đầu

Trên đường chéo `k`, ô **cao nhất-phải nhất** có `i` nhỏ nhất: `i = max(0, k - (n-1))`; ô **thấp nhất-trái nhất** có `i` lớn nhất: `i = min(k, m-1)`. Hai công thức này giúp ta biết đường chéo bắt đầu và kết thúc ở đâu.

In [4]:
for k in range(m + n - 1):
    top = max(0, k - (n - 1))            # i nhỏ nhất (ô cao nhất)
    bottom = min(k, m - 1)               # i lớn nhất (ô thấp nhất)
    print(f"k={k}: i từ {top} đến {bottom} (số ô {bottom - top + 1}), "
          f"ô thấp nhất = ({bottom}, {k - bottom}), ô cao nhất = ({top}, {k - top})")

k=0: i từ 0 đến 0 (số ô 1), ô thấp nhất = (0, 0), ô cao nhất = (0, 0)
k=1: i từ 0 đến 1 (số ô 2), ô thấp nhất = (1, 0), ô cao nhất = (0, 1)
k=2: i từ 0 đến 2 (số ô 3), ô thấp nhất = (2, 0), ô cao nhất = (0, 2)
k=3: i từ 0 đến 2 (số ô 3), ô thấp nhất = (2, 1), ô cao nhất = (0, 3)
k=4: i từ 1 đến 2 (số ô 2), ô thấp nhất = (2, 2), ô cao nhất = (1, 3)
k=5: i từ 2 đến 2 (số ô 1), ô thấp nhất = (2, 3), ô cao nhất = (2, 3)


### 3.5. `defaultdict(list)`, đảo list, `sorted` với `key`

- `defaultdict(list)`: từ điển tự tạo list rỗng khi truy cập khóa mới — tiện để **gom nhóm** theo khóa.
- `lst.reverse()` đảo tại chỗ; `lst[::-1]` tạo bản đảo.
- `sorted(items, key=f)` sắp xếp theo giá trị `f(item)`; `key` có thể trả **tuple** để sắp xếp theo nhiều tiêu chí.
- Biểu thức điều kiện: `a if cond else b`.

In [5]:
groups = defaultdict(list)
for i in range(2):
    for j in range(3):
        groups[i + j].append((i, j))
print(dict(groups))

lst = [1, 2, 3]
print(lst[::-1], lst)                # lst[::-1] không đổi lst
lst.reverse()
print(lst)                           # reverse() đổi tại chỗ

pairs = [(2, 5), (1, 9), (2, 1), (1, 3)]
print(sorted(pairs, key=lambda p: (p[0], -p[1])))        # tăng theo p[0], giảm theo p[1]
print("chẵn" if 4 % 2 == 0 else "lẻ")

{0: [(0, 0)], 1: [(0, 1), (1, 0)], 2: [(0, 2), (1, 1)], 3: [(1, 2)]}
[3, 2, 1] [1, 2, 3]
[3, 2, 1]
[(1, 9), (1, 3), (2, 5), (2, 1)]
chẵn


## 4. Ý tưởng nền tảng: tính tay

Với ma trận 3 × 4:

```
 a b c d        hàng 0:  a(0,0) b(0,1) c(0,2) d(0,3)
 e f g h        hàng 1:  e(1,0) f(1,1) g(1,2) h(1,3)
 i j k l        hàng 2:  i(2,0) j(2,1) k(2,2) l(2,3)
```

Các đường chéo `k = i + j` (từ 0 đến `3 + 4 - 2 = 5`, tức 6 đường):

| `k` | các ô (từ **hàng trên** xuống **hàng dưới**) | hướng đi | thứ tự đọc |
|---|---|---|---|
| 0 | a | (1 ô) | a |
| 1 | b(0,1), e(1,0) | **xuống-trái** (lẻ) | b, e |
| 2 | c(0,2), f(1,1), i(2,0) | **lên-phải** (chẵn): từ dưới lên | i, f, c |
| 3 | d(0,3), g(1,2), j(2,1) | xuống-trái (lẻ): từ trên xuống | d, g, j |
| 4 | h(1,3), k(2,2) | lên-phải (chẵn): từ dưới lên | k, h |
| 5 | l(2,3) | (1 ô) | l |

Kết quả: `a, b, e, i, f, c, d, g, j, k, h, l`.

> Quy tắc: mỗi đường chéo `k` luôn được liệt kê từ trên xuống (theo `i` tăng); nếu `k` **chẵn** thì **đảo ngược**.

In [6]:
mat_letters = [list("abcd"), list("efgh"), list("ijkl")]
order = []
for k in range(3 + 4 - 1):
    cells = [mat_letters[i][k - i] for i in range(3) if 0 <= k - i < 4]      # từ trên xuống
    if k % 2 == 0:
        cells.reverse()                                                       # k chẵn: đi lên
    print(k, cells)
    order += cells
print("".join(order))

0 ['a']
1 ['b', 'e']
2 ['i', 'f', 'c']
3 ['d', 'g', 'j']
4 ['k', 'h']
5 ['l']
abeifcdgjkhl


## 5. Cách 1 — Brute Force: sắp xếp mọi ô theo khóa

**Ý tưởng.** Tạo danh sách `(khóa, giá trị)` cho **mọi ô**, với khóa là một tuple `(k, t)`:

- `k = i + j` (đường chéo nào);
- `t = i` nếu `k` lẻ (đi xuống: `i` tăng dần), `t = -i` nếu `k` chẵn (đi lên: `i` giảm dần).

Sắp xếp rồi lấy giá trị theo thứ tự. Đơn giản, không cần nghĩ về biên, và là **oracle** tốt cho các cách khác.

**Code demo:**

In [7]:
def traverse_sort(mat):
    cells = []
    for i, row in enumerate(mat):
        for j, v in enumerate(row):
            k = i + j
            cells.append(((k, i if k % 2 else -i), v))
    cells.sort()
    return [v for _, v in cells]


for mat, expected in examples:
    got = traverse_sort(mat)
    print(got)
    assert got == expected

[1, 2, 4, 7, 5, 3, 6, 8, 9]
[1, 2, 3, 4]


**Code nộp LeetCode (Cách 1):**

In [8]:
class SolutionSort:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        cells = []
        for i, row in enumerate(mat):
            for j, v in enumerate(row):
                k = i + j
                cells.append(((k, i if k % 2 else -i), v))
        cells.sort()
        return [v for _, v in cells]


for mat, expected in examples:
    assert SolutionSort().findDiagonalOrder(mat) == expected
print("Cách 1 OK")

Cách 1 OK


**Độ phức tạp.** Tạo `m·n` mục, sắp xếp `O(m·n·log(m·n))` ⇒ **`O(mn log(mn))`** thời gian, **`O(mn)`** bộ nhớ phụ.

**Nút thắt.** Ta **sắp xếp** những thứ mà ta đã biết sẵn thứ tự (duyệt hàng-cột đã cho `i` tăng; chỉ cần nhóm theo `k`). Gom nhóm bằng từ điển sẽ bỏ được `log`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Nhóm theo `k = i + j`.** Duyệt ma trận theo hàng (i tăng, rồi j tăng). Trong mỗi nhóm `k`, các ô tự nhiên đến theo thứ tự **`i` tăng** (tức "từ trên xuống / xuống-trái") — đúng thứ tự của đường chéo **lẻ**. Đường chéo **chẵn** chỉ cần đảo ngược ⇒ **Cách 2** (`O(mn)`).

**Bước 2 — Đi thẳng, bật tại biên.** Giả lập một "con kiến" ở `(0,0)` đi lên-phải; khi sắp ra ngoài thì xử lý góc: lên-phải chạm **cạnh phải** thì đi **xuống**, chạm **cạnh trên** thì đi **sang phải**; xuống-trái chạm **cạnh dưới** thì đi **sang phải**, chạm **cạnh trái** thì đi **xuống**; rồi đổi hướng ⇒ **Cách 3** (`O(mn)` thời gian, `O(1)` bộ nhớ phụ).

**Bước 3 — Tính điểm bắt đầu từng đường chéo.** Với mỗi `k` ta biết ngay ô đầu: đường chéo chẵn (đi lên) bắt đầu ở `i = min(k, m-1)`, `j = k - i`; đường chéo lẻ (đi xuống) bắt đầu ở `j = min(k, n-1)`, `i = k - j`. Từ ô đầu đi cho tới khi ra ngoài ma trận ⇒ **Cách 4**, không cần xử lý "bật biên" vì mỗi đường chéo tự kết thúc.

## 7. Các cách tiếp theo

### Cách 2 — Gom theo `i + j` bằng từ điển rồi đảo xen kẽ

**Giải thích từng đoạn.**

1. `groups = defaultdict(list)`; duyệt `i`, `j` theo hàng; `groups[i + j].append(mat[i][j])`.
2. Với `k` từ `0` đến `m + n - 2`: nếu `k` **chẵn** thì đảo `groups[k]`; thêm vào kết quả.

Vì ta duyệt theo hàng nên trong `groups[k]` các phần tử đã sắp theo `i` tăng (thứ tự "xuống-trái").

In [9]:
def traverse_group(mat):
    m, n = len(mat), len(mat[0])
    groups = defaultdict(list)
    for i in range(m):
        for j in range(n):
            groups[i + j].append(mat[i][j])
    ans = []
    for k in range(m + n - 1):
        diag = groups[k]
        ans.extend(diag[::-1] if k % 2 == 0 else diag)
    return ans


for mat, expected in examples:
    assert traverse_group(mat) == expected
print(traverse_group([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 4, 7, 5, 3, 6, 8, 9]


In [10]:
class SolutionGroup:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        m, n = len(mat), len(mat[0])
        groups = defaultdict(list)
        for i in range(m):
            for j in range(n):
                groups[i + j].append(mat[i][j])
        ans: List[int] = []
        for k in range(m + n - 1):
            diag = groups[k]
            ans.extend(diag[::-1] if k % 2 == 0 else diag)
        return ans


for mat, expected in examples:
    assert SolutionGroup().findDiagonalOrder(mat) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp.** Duyệt `m·n` ô, đảo mỗi nhóm cũng tổng `O(m·n)` ⇒ **`O(mn)`** thời gian; bộ nhớ phụ **`O(mn)`** (từ điển chứa mọi phần tử).

### Cách 3 — Mô phỏng bước đi, bật khi chạm biên

**Giải thích từng đoạn.**

1. Trạng thái: vị trí `(i, j)` và cờ `up` (đang đi lên-phải?). Bắt đầu `(0,0)`, `up = True`.
2. Lặp đúng `m·n` lần: thêm `mat[i][j]` vào kết quả, rồi tính bước kế:
   - Đang `up` (đi `i-1, j+1`):
     - nếu `j == n-1` (chạm **cạnh phải**): đi **xuống** `i += 1`, `up = False`;
     - ngược lại nếu `i == 0` (chạm **cạnh trên**): đi **phải** `j += 1`, `up = False`;
     - ngược lại: `i -= 1; j += 1`.
   - Đang đi xuống (`i+1, j-1`):
     - nếu `i == m-1` (chạm **cạnh dưới**): `j += 1`, `up = True`;
     - ngược lại nếu `j == 0` (chạm **cạnh trái**): `i += 1`, `up = True`;
     - ngược lại: `i += 1; j -= 1`.

**Thứ tự kiểm tra rất quan trọng ở góc.** Khi `up` ở góc trên-phải `(0, n-1)` thì cả "cạnh phải" lẫn "cạnh trên" đều đúng; phải ưu tiên **cạnh phải** (đi xuống), vì đi sang phải sẽ ra ngoài. Tương tự khi đi xuống ở góc dưới-trái `(m-1, 0)` phải ưu tiên **cạnh dưới** (đi sang phải).

In [11]:
def traverse_sim(mat):
    m, n = len(mat), len(mat[0])
    ans = []
    i = j = 0
    up = True
    for _ in range(m * n):
        ans.append(mat[i][j])
        if up:
            if j == n - 1:          # chạm cạnh phải (ưu tiên trước cạnh trên)
                i += 1
                up = False
            elif i == 0:            # chạm cạnh trên
                j += 1
                up = False
            else:
                i -= 1
                j += 1
        else:
            if i == m - 1:          # chạm cạnh dưới (ưu tiên trước cạnh trái)
                j += 1
                up = True
            elif j == 0:            # chạm cạnh trái
                i += 1
                up = True
            else:
                i += 1
                j -= 1
    return ans


for mat, expected in examples:
    assert traverse_sim(mat) == expected
print(traverse_sim([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 4, 7, 5, 3, 6, 8, 9]


In [12]:
class SolutionSim:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        m, n = len(mat), len(mat[0])
        ans: List[int] = []
        i = j = 0
        up = True
        for _ in range(m * n):
            ans.append(mat[i][j])
            if up:
                if j == n - 1:
                    i += 1
                    up = False
                elif i == 0:
                    j += 1
                    up = False
                else:
                    i -= 1
                    j += 1
            else:
                if i == m - 1:
                    j += 1
                    up = True
                elif j == 0:
                    i += 1
                    up = True
                else:
                    i += 1
                    j -= 1
        return ans


for mat, expected in examples:
    assert SolutionSim().findDiagonalOrder(mat) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp.** Đúng `m·n` bước, mỗi bước `O(1)` ⇒ **`O(mn)`** thời gian; bộ nhớ phụ **`O(1)`** (ngoài mảng kết quả). Nhược điểm: nhiều nhánh biên dễ nhầm.

### Cách 4 — Duyệt từng đường chéo, điểm bắt đầu tính bằng công thức (chốt nộp)

**Giải thích từng đoạn.**

1. Với mỗi `k` từ `0` đến `m + n - 2`:
2. Nếu `k` **chẵn** (đi lên-phải): bắt đầu ở ô thấp nhất `i = min(k, m - 1)`, `j = k - i`. Lặp: thêm `mat[i][j]`, `i -= 1`, `j += 1` cho tới khi `i < 0` hoặc `j >= n`.
3. Nếu `k` **lẻ** (đi xuống-trái): bắt đầu ở ô cao nhất `j = min(k, n - 1)`, `i = k - j`. Lặp: thêm `mat[i][j]`, `i += 1`, `j -= 1` cho tới khi `i >= m` hoặc `j < 0`.

Mỗi đường chéo tự dừng khi ra khỏi ma trận, nên **không cần xử lý trường hợp "góc"** — các góc được "ăn" luôn bởi `min(...)`.

In [13]:
def traverse_diag(mat):
    m, n = len(mat), len(mat[0])
    ans = []
    for k in range(m + n - 1):
        if k % 2 == 0:                                   # đi lên-phải, bắt đầu ở ô thấp nhất
            i = min(k, m - 1)
            j = k - i
            while i >= 0 and j < n:
                ans.append(mat[i][j])
                i -= 1
                j += 1
        else:                                            # đi xuống-trái, bắt đầu ở ô cao nhất
            j = min(k, n - 1)
            i = k - j
            while j >= 0 and i < m:
                ans.append(mat[i][j])
                i += 1
                j -= 1
    return ans


for mat, expected in examples:
    assert traverse_diag(mat) == expected
print(traverse_diag([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 4, 7, 5, 3, 6, 8, 9]


In [14]:
class SolutionDiag:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        m, n = len(mat), len(mat[0])
        ans: List[int] = []
        for k in range(m + n - 1):
            if k % 2 == 0:
                i = min(k, m - 1)
                j = k - i
                while i >= 0 and j < n:
                    ans.append(mat[i][j])
                    i -= 1
                    j += 1
            else:
                j = min(k, n - 1)
                i = k - j
                while j >= 0 and i < m:
                    ans.append(mat[i][j])
                    i += 1
                    j -= 1
        return ans


for mat, expected in examples:
    assert SolutionDiag().findDiagonalOrder(mat) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp.** Tổng số ô thăm là `m·n` ⇒ **`O(mn)`** thời gian; bộ nhớ phụ **`O(1)`** (ngoài kết quả). Code sáng sủa, ít nhánh biên.

## 8. TRACE từng bước

**Trace Cách 4** với ma trận `mat = [[1,2,3,4],[5,6,7,8],[9,10,11,12]]` (`m = 3`, `n = 4`, có `6` đường chéo):

| `k` | hướng | điểm bắt đầu `(i, j)` | các ô thăm | giá trị |
|---|---|---|---|---|
| 0 | lên | `i = min(0,2) = 0`, `j = 0` | (0,0) | 1 |
| 1 | xuống | `j = min(1,3) = 1`, `i = 0` | (0,1), (1,0) | 2, 5 |
| 2 | lên | `i = min(2,2) = 2`, `j = 0` | (2,0), (1,1), (0,2) | 9, 6, 3 |
| 3 | xuống | `j = min(3,3) = 3`, `i = 0` | (0,3), (1,2), (2,1) | 4, 7, 10 |
| 4 | lên | `i = min(4,2) = 2`, `j = 2` | (2,2), (1,3) | 11, 8 |
| 5 | xuống | `j = min(5,3) = 3`, `i = 2` | (2,3) | 12 |

Kết quả: `[1, 2, 5, 9, 6, 3, 4, 7, 10, 11, 8, 12]`.

In [15]:
def trace_diag(mat):
    m, n = len(mat), len(mat[0])
    print(f"m={m}, n={n}, số đường chéo = m+n-1 = {m + n - 1}")
    ans = []
    for k in range(m + n - 1):
        visited = []
        if k % 2 == 0:
            i = min(k, m - 1); j = k - i
            start = (i, j)
            while i >= 0 and j < n:
                visited.append((i, j)); ans.append(mat[i][j])
                i -= 1; j += 1
            direction = "lên-phải "
        else:
            j = min(k, n - 1); i = k - j
            start = (i, j)
            while j >= 0 and i < m:
                visited.append((i, j)); ans.append(mat[i][j])
                i += 1; j -= 1
            direction = "xuống-trái"
        print(f"k={k} {direction} bắt đầu {start}: {visited} -> {[mat[a][b] for a, b in visited]}")
    print("kết quả:", ans)
    return ans


r = trace_diag([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])

m=3, n=4, số đường chéo = m+n-1 = 6
k=0 lên-phải  bắt đầu (0, 0): [(0, 0)] -> [1]
k=1 xuống-trái bắt đầu (0, 1): [(0, 1), (1, 0)] -> [2, 5]
k=2 lên-phải  bắt đầu (2, 0): [(2, 0), (1, 1), (0, 2)] -> [9, 6, 3]
k=3 xuống-trái bắt đầu (0, 3): [(0, 3), (1, 2), (2, 1)] -> [4, 7, 10]
k=4 lên-phải  bắt đầu (2, 2): [(2, 2), (1, 3)] -> [11, 8]
k=5 xuống-trái bắt đầu (2, 3): [(2, 3)] -> [12]
kết quả: [1, 2, 5, 9, 6, 3, 4, 7, 10, 11, 8, 12]


In [16]:
def trace_sim(mat):
    m, n = len(mat), len(mat[0])
    i = j = 0
    up = True
    ans = []
    print(f"{'bước':>4} | {'(i,j)':>7} | {'giá trị':>7} | hướng | hành động")
    for step in range(m * n):
        ans.append(mat[i][j])
        cur = (i, j)
        d = "lên  " if up else "xuống"
        if up:
            if j == n - 1:
                why = "chạm cạnh phải -> xuống 1, đổi hướng"; i += 1; up = False
            elif i == 0:
                why = "chạm cạnh trên -> sang phải 1, đổi hướng"; j += 1; up = False
            else:
                why = "đi tiếp lên-phải"; i -= 1; j += 1
        else:
            if i == m - 1:
                why = "chạm cạnh dưới -> sang phải 1, đổi hướng"; j += 1; up = True
            elif j == 0:
                why = "chạm cạnh trái -> xuống 1, đổi hướng"; i += 1; up = True
            else:
                why = "đi tiếp xuống-trái"; i += 1; j -= 1
        print(f"{step + 1:>4} | {str(cur):>7} | {mat[cur[0]][cur[1]]:>7} | {d} | {why}")
    print("kết quả:", ans)
    return ans


r = trace_sim([[1, 2, 3], [4, 5, 6]])

bước |   (i,j) | giá trị | hướng | hành động
   1 |  (0, 0) |       1 | lên   | chạm cạnh trên -> sang phải 1, đổi hướng
   2 |  (0, 1) |       2 | xuống | đi tiếp xuống-trái
   3 |  (1, 0) |       4 | xuống | chạm cạnh dưới -> sang phải 1, đổi hướng
   4 |  (1, 1) |       5 | lên   | đi tiếp lên-phải
   5 |  (0, 2) |       3 | lên   | chạm cạnh phải -> xuống 1, đổi hướng
   6 |  (1, 2) |       6 | xuống | chạm cạnh dưới -> sang phải 1, đổi hướng
kết quả: [1, 2, 4, 5, 3, 6]


## 9. Code nộp LeetCode

Ta nộp Cách 4 (từng đường chéo, điểm bắt đầu theo công thức): gọn, `O(mn)` thời gian, `O(1)` bộ nhớ phụ.

In [17]:
class Solution:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        m, n = len(mat), len(mat[0])
        ans: List[int] = []
        for k in range(m + n - 1):               # k = i + j: có đúng m + n - 1 đường chéo
            if k % 2 == 0:                       # k chẵn: đi lên-phải từ ô thấp nhất
                i = min(k, m - 1)
                j = k - i
                while i >= 0 and j < n:
                    ans.append(mat[i][j])
                    i -= 1
                    j += 1
            else:                                # k lẻ: đi xuống-trái từ ô cao nhất
                j = min(k, n - 1)
                i = k - j
                while j >= 0 and i < m:
                    ans.append(mat[i][j])
                    i += 1
                    j -= 1
        return ans


for mat, expected in examples:
    got = Solution().findDiagonalOrder(mat)
    print(mat, "->", got)
    assert got == expected

[[1, 2, 3], [4, 5, 6], [7, 8, 9]] -> [1, 2, 4, 7, 5, 3, 6, 8, 9]
[[1, 2], [3, 4]] -> [1, 2, 3, 4]


## 10. Vì sao đúng

**Phân hoạch.** Mỗi ô `(i, j)` thuộc đúng một đường chéo `k = i + j ∈ [0, m+n-2]`. Vòng `for k` duyệt đủ mọi `k`, nên mọi ô được xét; ta cần chứng minh mỗi ô được thêm **đúng một lần** và đúng thứ tự.

**Điểm bắt đầu và kết thúc (đường chéo chẵn).** Các ô của đường chéo `k` có `i ∈ [max(0, k-(n-1)), min(k, m-1)]`. Điểm bắt đầu `i = min(k, m-1)` là **ô có `i` lớn nhất**, và `j = k - i ≥ 0`. Đi `i -= 1, j += 1` đi qua đúng các ô của đường chéo theo `i` giảm; dừng khi `i < 0` hoặc `j >= n` — tức ngay sau ô có `i` nhỏ nhất (nếu `k ≤ n-1` thì `i` xuống `0` rồi `-1`; nếu `k > n-1` thì `j` chạm `n` trước). Do đó mọi ô của đường chéo được thăm đúng một lần.

**Đường chéo lẻ.** Tương tự với vai trò `i ↔ j`: bắt đầu `j = min(k, n-1)` (ô có `i` nhỏ nhất, vì `i = k - j`), đi `i += 1, j -= 1` theo `i` tăng; dừng khi `j < 0` hoặc `i >= m`.

**Hướng đúng đề.** Đề yêu cầu đường chéo đầu tiên (k=0) có 1 ô, k=1 đi xuống-trái (từ `(0,1)` sang `(1,0)`), k=2 đi lên-phải, ... — tức **chẵn đi lên, lẻ đi xuống**. Cách 4 làm đúng như vậy; Cách 2, 3 và oracle sắp xếp cho cùng kết quả (kiểm chứng ở mục 12).

## 11. Time / Space Complexity

Gọi `N = m · n` là số ô.

| Cách | Time | Space phụ (ngoài kết quả) |
|---|---|---|
| 1. Sắp xếp theo khóa | `O(N log N)` | `O(N)` |
| 2. Gom theo `i + j` | `O(N)` | `O(N)` |
| 3. Mô phỏng bật biên | `O(N)` | `O(1)` |
| 4. Từng đường chéo | `O(N)` | `O(1)` |

Không thể nhanh hơn `O(N)` vì phải xuất mọi phần tử.

## 12. Bộ test

Oracle độc lập: `traverse_sort` (Cách 1, định nghĩa qua khóa sắp xếp) và một oracle nữa `oracle_walk` đi theo **định nghĩa hình học** (đi lên-phải, ra ngoài thì chuyển) ở dạng khác. Đối chiếu bốn cách với oracle trên rất nhiều ma trận, gồm mọi kích thước nhỏ.

In [18]:
def oracle_walk(mat):
    """Oracle thứ hai: duyệt theo định nghĩa đường chéo bằng cách thử ô kế tiếp và 'trượt' khi ra ngoài."""
    m, n = len(mat), len(mat[0])
    seen = [[False] * n for _ in range(m)]
    i = j = 0
    d = (-1, 1)
    ans = []
    while len(ans) < m * n:
        ans.append(mat[i][j])
        seen[i][j] = True
        ni, nj = i + d[0], j + d[1]
        if 0 <= ni < m and 0 <= nj < n:
            i, j = ni, nj
            continue
        # ra ngoài: dịch sang đường chéo kế tiếp rồi đổi hướng
        d = (-d[0], -d[1])
        # ô đầu của đường chéo kế tiếp: thử phải rồi xuống (hoặc xuống rồi phải) theo hướng mới
        cands = [(i, j + 1), (i + 1, j)] if d == (1, -1) else [(i + 1, j), (i, j + 1)]
        for ci, cj in cands:
            if 0 <= ci < m and 0 <= cj < n and not seen[ci][cj]:
                i, j = ci, cj
                break
    return ans


def make_mat(m, n, rng, lo=-10 ** 5, hi=10 ** 5):
    return [[rng.randint(lo, hi) for _ in range(n)] for _ in range(m)]


def index_mat(m, n):
    return [[i * n + j for j in range(n)] for i in range(m)]


SOLUTIONS = [Solution(), SolutionSort(), SolutionGroup(), SolutionSim(), SolutionDiag()]
print(oracle_walk([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 4, 7, 5, 3, 6, 8, 9]


In [19]:
# 12.1 Ví dụ đề + edge case
cases = [
    ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [1, 2, 4, 7, 5, 3, 6, 8, 9]),
    ([[1, 2], [3, 4]], [1, 2, 3, 4]),
    ([[5]], [5]),                                   # 1 x 1
    ([[1, 2, 3, 4]], [1, 2, 3, 4]),                 # 1 hàng
    ([[1], [2], [3], [4]], [1, 2, 3, 4]),           # 1 cột
    ([[1, 2, 3], [4, 5, 6]], [1, 2, 4, 5, 3, 6]),   # 2 x 3
    ([[1, 2], [3, 4], [5, 6]], [1, 2, 3, 5, 4, 6]), # 3 x 2
    ([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]], [1, 2, 5, 9, 6, 3, 4, 7, 10, 11, 8, 12]),
    ([[-100000, 100000], [0, -1]], [-100000, 100000, 0, -1]),
]
for mat, expected in cases:
    assert traverse_sort(mat) == expected, (mat, traverse_sort(mat))
    assert oracle_walk(mat) == expected, (mat, oracle_walk(mat))
    for sol in SOLUTIONS:
        got = sol.findDiagonalOrder([row[:] for row in mat])
        assert got == expected, (type(sol).__name__, mat, got, expected)
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp")

OK: 9 test x 5 lớp


In [20]:
# 12.2 Mọi kích thước 1 <= m, n <= 12 với ma trận chỉ số (mỗi ô một giá trị duy nhất)
count = 0
for m in range(1, 13):
    for n in range(1, 13):
        mat = index_mat(m, n)
        exp = traverse_sort(mat)
        assert oracle_walk(mat) == exp, (m, n)
        for sol in SOLUTIONS:
            assert sol.findDiagonalOrder(mat) == exp, (type(sol).__name__, m, n)
        assert sorted(exp) == list(range(m * n))              # mỗi ô đúng một lần
        count += 1
print("OK: đối chiếu", count, "kích thước")

OK: đối chiếu 144 kích thước


In [21]:
# 12.3 Ngẫu nhiên có seed: m * n <= 10^4, giá trị trong [-10^5, 10^5]
rng = random.Random(498)
for _ in range(300):
    m = rng.randint(1, 40)
    n = rng.randint(1, max(1, min(40, 10 ** 4 // m)))
    mat = make_mat(m, n, rng)
    exp = traverse_sort(mat)
    assert oracle_walk(mat) == exp
    for sol in SOLUTIONS:
        assert sol.findDiagonalOrder(mat) == exp, (type(sol).__name__, m, n)
print("OK: 300 ma trận ngẫu nhiên")

OK: 300 ma trận ngẫu nhiên


In [22]:
# 12.4 Kích thước lớn nhất: 1 x 10^4, 10^4 x 1, 100 x 100, 2 x 5000, 50 x 200
rng = random.Random(4981)
for m, n in [(1, 10 ** 4), (10 ** 4, 1), (100, 100), (2, 5000), (5000, 2), (50, 200), (200, 50)]:
    mat = make_mat(m, n, rng)
    t0 = time.perf_counter()
    exp = SolutionDiag().findDiagonalOrder(mat)
    t1 = time.perf_counter() - t0
    assert len(exp) == m * n
    for sol in SOLUTIONS:
        assert sol.findDiagonalOrder(mat) == exp, (type(sol).__name__, m, n)
    assert exp == traverse_sort(mat)
    print(f"{m:>5} x {n:<5}: OK ({t1 * 1000:.1f} ms cho Cách 4)")

    1 x 10000: OK (1.6 ms cho Cách 4)
10000 x 1    : OK (1.8 ms cho Cách 4)
  100 x 100  : OK (0.7 ms cho Cách 4)
    2 x 5000 : OK (1.2 ms cho Cách 4)
 5000 x 2    : OK (1.3 ms cho Cách 4)
   50 x 200  : OK (0.7 ms cho Cách 4)
  200 x 50   : OK (0.7 ms cho Cách 4)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm sai số đường chéo (`m + n - 2` thay vì `m + n - 1`)

In [23]:
def wrong_count(mat):
    m, n = len(mat), len(mat[0])
    ans = []
    for k in range(m + n - 2):                               # SAI: thiếu 1 đường chéo (k = m + n - 2)
        if k % 2 == 0:
            i = min(k, m - 1); j = k - i
            while i >= 0 and j < n:
                ans.append(mat[i][j]); i -= 1; j += 1
        else:
            j = min(k, n - 1); i = k - j
            while j >= 0 and i < m:
                ans.append(mat[i][j]); i += 1; j -= 1
    return ans


mat = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("sai :", wrong_count(mat), "(thiếu phần tử cuối 9)")
print("đúng:", Solution().findDiagonalOrder(mat))

sai : [1, 2, 4, 7, 5, 3, 6, 8] (thiếu phần tử cuối 9)
đúng: [1, 2, 4, 7, 5, 3, 6, 8, 9]


**Vì sao sai?** `k = i + j` chạy từ `0` đến `m + n - 2`, tức **`m + n - 1`** giá trị (ví dụ `3 x 3`: `k = 0..4`, 5 đường). `range(m + n - 2)` chỉ tới `m + n - 3` nên bỏ mất đường chéo cuối (ô góc dưới-phải). Ngược lại `range(m + n)` thêm một đường chéo rỗng: với cách tính điểm đầu bằng `min` thì vô hại nhưng với `groups[k]` có thể gây `KeyError` / khóa thừa. Luôn `range(m + n - 1)`.

### Lỗi 2: kiểm tra biên sai thứ tự ở góc (mô phỏng)

In [24]:
def wrong_order(mat):
    m, n = len(mat), len(mat[0])
    ans = []
    i = j = 0
    up = True
    for _ in range(m * n):
        ans.append(mat[i][j])
        if up:
            if i == 0:                    # SAI: kiểm tra cạnh trên TRƯỚC cạnh phải
                j += 1
                up = False
            elif j == n - 1:
                i += 1
                up = False
            else:
                i -= 1; j += 1
        else:
            if j == 0:                    # SAI: kiểm tra cạnh trái TRƯỚC cạnh dưới
                i += 1
                up = True
            elif i == m - 1:
                j += 1
                up = True
            else:
                i += 1; j -= 1
    return ans


mat = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
try:
    print("sai :", wrong_order(mat))
except IndexError as e:
    print("sai : IndexError ->", e)
print("đúng:", Solution().findDiagonalOrder(mat))

sai : IndexError -> list index out of range
đúng: [1, 2, 4, 7, 5, 3, 6, 8, 9]


**Vì sao sai?** Ở góc trên-phải `(0, 2)` đang đi lên: cả "cạnh trên" lẫn "cạnh phải" đúng. Phải ưu tiên **cạnh phải** (đi xuống). Nếu ưu tiên cạnh trên sẽ `j += 1` ⇒ `j = 3` ra ngoài ma trận (IndexError hoặc đọc nhầm). Tương tự ở góc dưới-trái khi đi xuống: ưu tiên **cạnh dưới** trước cạnh trái. Quy tắc: **kiểm tra cạnh "cuối đường" (phải/dưới) trước**.

### Lỗi 3: nhầm `m` ↔ `n` khi tính điểm bắt đầu

In [25]:
def wrong_start(mat):
    m, n = len(mat), len(mat[0])
    ans = []
    for k in range(m + n - 1):
        if k % 2 == 0:
            i = min(k, n - 1)                 # SAI: phải là m - 1 (i chạy theo hàng)
            j = k - i
            while i >= 0 and j < n:
                if i < m:
                    ans.append(mat[i][j])
                i -= 1; j += 1
        else:
            j = min(k, m - 1)                 # SAI: phải là n - 1
            i = k - j
            while j >= 0 and i < m:
                if j < n:
                    ans.append(mat[i][j])
                i += 1; j -= 1
    return ans


mat = [[1, 2, 3, 4], [5, 6, 7, 8]]            # 2 x 4: m != n
print("sai :", wrong_start(mat))
print("đúng:", Solution().findDiagonalOrder(mat))

sai : [1, 2, 5, 6, 3, 8]
đúng: [1, 2, 5, 6, 3, 4, 7, 8]


**Vì sao sai?** Với ma trận không vuông, `i` bị chặn bởi `m - 1` còn `j` bị chặn bởi `n - 1`. Đổi chỗ cho nhau làm điểm đầu sai và **bỏ sót** ô. Hãy thử luôn ma trận không vuông (2 × 4, 4 × 2, 1 × n).

### Lỗi 4: đảo sai đường chéo (chẵn ↔ lẻ)

Đề: đường chéo đầu tiên `k = 0` có một ô; đường chéo `k = 1` đi **xuống-trái** (`2` rồi `4` ở ví dụ 1 là `[1,2,4,...]`). Nếu ta đảo nhầm (đảo các `k` lẻ) thì ví dụ 1 ra `[1,4,2,...]` — sai ngay ở phần tử thứ hai.

In [26]:
def wrong_parity(mat):
    m, n = len(mat), len(mat[0])
    groups = defaultdict(list)
    for i in range(m):
        for j in range(n):
            groups[i + j].append(mat[i][j])
    ans = []
    for k in range(m + n - 1):
        ans.extend(groups[k][::-1] if k % 2 == 1 else groups[k])         # SAI: đảo k lẻ
    return ans


mat = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("sai :", wrong_parity(mat))
print("đúng:", Solution().findDiagonalOrder(mat))

sai : [1, 4, 2, 3, 5, 7, 8, 6, 9]
đúng: [1, 2, 4, 7, 5, 3, 6, 8, 9]


### Lỗi 5: tạo danh sách lồng bằng `[[]] * k` (cùng tham chiếu)

In [27]:
m, n = 2, 3
groups = [[]] * (m + n - 1)                   # SAI: cả 4 phần tử là CÙNG một list
mat = [[1, 2, 3], [4, 5, 6]]
for i in range(m):
    for j in range(n):
        groups[i + j].append(mat[i][j])
print("sai :", groups)

groups = [[] for _ in range(m + n - 1)]       # ĐÚNG: mỗi phần tử một list riêng
for i in range(m):
    for j in range(n):
        groups[i + j].append(mat[i][j])
print("đúng:", groups)

sai : [[1, 2, 3, 4, 5, 6], [1, 2, 3, 4, 5, 6], [1, 2, 3, 4, 5, 6], [1, 2, 3, 4, 5, 6]]
đúng: [[1], [2, 4], [3, 5], [6]]


**Vì sao sai?** `[[]] * k` lặp lại **cùng một đối tượng** list `k` lần; thêm vào một phần tử là thêm vào tất cả. Dùng list comprehension `[[] for _ in range(k)]` hoặc `defaultdict(list)`.

## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "duyệt ma trận theo đường chéo / zigzag" ⇒ nghĩ ngay tới **`i + j` (đường chéo phụ)** hoặc **`i - j` (đường chéo chính)** là hằng số trên một đường.
- **Số đường chéo** là `m + n - 1`. Số ô trên đường chéo `k` là `min(k, m-1) - max(0, k-(n-1)) + 1`.
- **Hai kiểu viết:** (a) mô phỏng bước đi + bật biên (nhiều nhánh biên, dễ lỗi); (b) lặp theo đường chéo với điểm bắt đầu tính bằng công thức (gọn). Khi phỏng vấn, (b) thường ít lỗi hơn.
- **Ưu tiên cạnh "cuối đường"** khi viết mô phỏng: cạnh phải khi đi lên, cạnh dưới khi đi xuống.
- **Test ma trận không vuông**, một hàng, một cột — đó là nơi lỗi thường xuất hiện.
- Tự kiểm bằng ma trận chỉ số (`mat[i][j] = i*n + j`): mỗi ô một số duy nhất, kết quả phải là hoán vị của `0..mn-1`.

## 15. Biến thể

1. **Diagonal Traverse II (1424):** mảng "lởm chởm" (mỗi hàng độ dài khác nhau), mỗi đường chéo đọc từ dưới lên, **không** zigzag.
2. **Spiral Matrix (54):** duyệt xoắn ốc — cũng là mô phỏng với "bật biên".
3. **Zigzag Conversion (6):** duyệt chuỗi theo hình chữ Z.
4. **Sort the Matrix Diagonally (1329):** sắp xếp từng đường chéo **chính** (`i - j` không đổi).
5. **Toeplitz Matrix (766):** kiểm tra mọi đường chéo chính có cùng giá trị.
6. **Tổng từng đường chéo phụ:** `sums[i + j] += mat[i][j]`.

In [28]:
# Biến thể 6 và 5: tổng đường chéo phụ, và kiểm tra Toeplitz
def anti_diagonal_sums(mat):
    m, n = len(mat), len(mat[0])
    sums = [0] * (m + n - 1)
    for i in range(m):
        for j in range(n):
            sums[i + j] += mat[i][j]
    return sums


def is_toeplitz(mat):
    return all(mat[i][j] == mat[i - 1][j - 1] for i in range(1, len(mat)) for j in range(1, len(mat[0])))


print(anti_diagonal_sums([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))     # [1, 6, 15, 14, 9]
print(is_toeplitz([[1, 2, 3], [4, 1, 2], [7, 4, 1]]), is_toeplitz([[1, 2], [2, 2]]))

[1, 6, 15, 14, 9]
True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp theo khóa | khóa `(i+j, ±i)` | `O(N log N)` | `O(N)` | làm oracle, ý tưởng nhanh |
| 2. Gom theo `i+j` | từ điển + đảo chẵn | `O(N)` | `O(N)` | dễ hiểu, dễ chứng minh |
| 3. Mô phỏng bật biên | con kiến + cờ hướng | `O(N)` | `O(1)` | khi đề không cho công thức rõ; cần cẩn thận góc |
| 4. Từng đường chéo | điểm đầu bằng `min`, đi tới khi ra ngoài | `O(N)` | `O(1)` | **nộp bài**: ngắn, ít lỗi |

In [29]:
rng = random.Random(1)
mat = make_mat(100, 100, rng)
print("100 x 100:")
for name, sol in [("sort", SolutionSort()), ("group", SolutionGroup()), ("sim", SolutionSim()), ("diag", SolutionDiag())]:
    t0 = time.perf_counter()
    for _ in range(20):
        sol.findDiagonalOrder(mat)
    print(f"{name:<6}: {(time.perf_counter() - t0) / 20 * 1000:.2f} ms")

100 x 100:


sort  : 5.67 ms
group : 0.55 ms
sim   : 0.80 ms
diag  : 0.65 ms


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) số đường chéo của ma trận `4 x 7`; (b) `Solution().findDiagonalOrder([[1,2,3]])`; (c) `Solution().findDiagonalOrder([[1],[2],[3]])`; (d) `min(5, 3 - 1)` và `5 - min(5, 3 - 1)` khi `m = 3`, `n = 4`, `k = 5`.

**Bài 2 (tính tay).** Với `mat = [[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]` (4 × 4), viết thứ tự duyệt zigzag. Gợi ý: chẵn đi lên, lẻ đi xuống.

**Bài 3 (điền TODO).** Hoàn thành `diag_todo(mat)` theo Cách 4.

In [30]:
def diag_todo(mat):
    # TODO 1: m, n = len(mat), len(mat[0]); ans = []
    # TODO 2: for k in range(m + n - 1):
    # TODO 3: nếu k chẵn: i = min(k, m - 1), j = k - i; while i >= 0 and j < n: thêm mat[i][j], i -= 1, j += 1
    # TODO 4: nếu k lẻ : j = min(k, n - 1), i = k - j; while j >= 0 and i < m: thêm mat[i][j], i += 1, j -= 1
    # TODO 5: return ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `sim_again(mat)` theo Cách 3 (mô phỏng bật biên) **không nhìn lại** mục 7. Kiểm tra với ma trận `3 x 3`, `2 x 3`, `1 x 4`, `4 x 1`.

**Bài 5 (biến thể / mini challenge).** Cho mảng "lởm chởm" `nums` (mỗi hàng có độ dài khác nhau). Viết `diagonal_order_jagged(nums)` trả về các phần tử theo từng đường chéo phụ `i + j = k`, mỗi đường chéo đọc **từ hàng dưới lên hàng trên** (không zigzag), `k` tăng dần (đúng LeetCode 1424). Kiểm chứng bằng cách sắp xếp mọi ô theo `(i + j, -i)`.

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ).** Bài 1: số đường chéo là `m + n - 1`. Bài 3: chép đúng ba bước của Cách 4, đừng quên `min`.

**Mức 2 (vừa).** Bài 2: các đường chéo `k = 0..6`; `k` chẵn đọc từ **dưới lên**, `k` lẻ đọc từ **trên xuống**. Bài 4: nhớ ưu tiên cạnh phải khi đi lên, cạnh dưới khi đi xuống.

**Mức 3 (sát đáp án).** Bài 5: gom `groups[i + j].append(nums[i][j])` duyệt `i` **giảm dần** (từ hàng cuối lên) thì mỗi nhóm đã ở thứ tự "dưới lên"; sau đó nối các nhóm theo `k` tăng dần.

## 19. Đáp án tham khảo cho bài tập

In [31]:
# Bài 1
assert 4 + 7 - 1 == 10
assert Solution().findDiagonalOrder([[1, 2, 3]]) == [1, 2, 3]
assert Solution().findDiagonalOrder([[1], [2], [3]]) == [1, 2, 3]
assert min(5, 3 - 1) == 2 and 5 - min(5, 3 - 1) == 3

# Bài 2: 4 x 4
m16 = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
expected16 = [1, 2, 5, 9, 6, 3, 4, 7, 10, 13, 14, 11, 8, 12, 15, 16]
assert Solution().findDiagonalOrder(m16) == expected16
assert traverse_sort(m16) == expected16
print("bài 1-2 OK")

bài 1-2 OK


In [32]:
# Bài 3
def diag_todo_answer(mat):
    m, n = len(mat), len(mat[0])
    ans = []
    for k in range(m + n - 1):
        if k % 2 == 0:
            i = min(k, m - 1)
            j = k - i
            while i >= 0 and j < n:
                ans.append(mat[i][j])
                i -= 1
                j += 1
        else:
            j = min(k, n - 1)
            i = k - j
            while j >= 0 and i < m:
                ans.append(mat[i][j])
                i += 1
                j -= 1
    return ans


for mat, expected in examples:
    assert diag_todo_answer(mat) == expected
assert diag_todo_answer(m16) == expected16
print("bài 3 OK")

bài 3 OK


In [33]:
# Bài 4
def sim_again(mat):
    m, n = len(mat), len(mat[0])
    i = j = 0
    up = True
    ans = []
    for _ in range(m * n):
        ans.append(mat[i][j])
        if up:
            if j == n - 1:
                i += 1; up = False
            elif i == 0:
                j += 1; up = False
            else:
                i -= 1; j += 1
        else:
            if i == m - 1:
                j += 1; up = True
            elif j == 0:
                i += 1; up = True
            else:
                i += 1; j -= 1
    return ans


for m_, n_ in [(3, 3), (2, 3), (1, 4), (4, 1), (5, 2), (1, 1)]:
    mm = index_mat(m_, n_)
    assert sim_again(mm) == traverse_sort(mm), (m_, n_)
print("bài 4 OK")

bài 4 OK


In [34]:
# Bài 5: LeetCode 1424 (mỗi đường chéo đọc từ dưới lên, không zigzag)
def diagonal_order_jagged(nums):
    groups = defaultdict(list)
    for i in range(len(nums) - 1, -1, -1):             # từ hàng cuối lên
        for j, v in enumerate(nums[i]):
            groups[i + j].append(v)
    ans = []
    for k in range(len(groups)):                       # các khóa là 0..K liên tục
        ans.extend(groups[k])
    return ans


def jagged_oracle(nums):
    cells = [((i + j, -i), v) for i, row in enumerate(nums) for j, v in enumerate(row)]
    cells.sort()
    return [v for _, v in cells]


assert diagonal_order_jagged([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 4, 2, 7, 5, 3, 8, 6, 9]
assert diagonal_order_jagged([[1, 2, 3, 4, 5], [6, 7], [8], [9, 10, 11], [12, 13, 14, 15, 16]]) == \
    [1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]
rng = random.Random(4982)
for _ in range(300):
    rows = rng.randint(1, 8)
    nums = [[rng.randint(-5, 5) for _ in range(rng.randint(1, 8))] for _ in range(rows)]
    assert diagonal_order_jagged(nums) == jagged_oracle(nums), nums
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (từng đường chéo `k = i + j`)

```python
class Solution:
    def findDiagonalOrder(self, mat: List[List[int]]) -> List[int]:
        m, n = len(mat), len(mat[0])
        ans = []
        for k in range(m + n - 1):
            if k % 2 == 0:                       # đi lên-phải
                i = min(k, m - 1)
                j = k - i
                while i >= 0 and j < n:
                    ans.append(mat[i][j])
                    i -= 1
                    j += 1
            else:                                # đi xuống-trái
                j = min(k, n - 1)
                i = k - j
                while j >= 0 and i < m:
                    ans.append(mat[i][j])
                    i += 1
                    j -= 1
        return ans
```

**Checklist**

- [ ] Hiểu: đường chéo phụ `i + j = k`, có `m + n - 1` đường, chẵn đi lên / lẻ đi xuống?
- [ ] Điểm bắt đầu dùng đúng `m` cho `i` và `n` cho `j` (ma trận không vuông)?
- [ ] Nếu mô phỏng: kiểm tra cạnh phải/dưới **trước** cạnh trên/trái?
- [ ] Không dùng `[[]] * k`?
- [ ] Đã thử `1 x 1`, `1 x n`, `m x 1`, `2 x 2`, `2 x 3`, `3 x 2`, `m*n = 10^4`?
- [ ] Đã so với oracle sắp xếp theo khóa `(i + j, ±i)` trên ma trận chỉ số?